# Qwen-Image-2.1 via stable-diffusion.cpp — Colab T4

Jalankan **Qwen-Image-2.1** di Colab free (T4) pakai
[stable-diffusion.cpp](https://github.com/leejet/stable-diffusion.cpp)
(day-0 support Qwen-Image-2.1 sejak 2026-09-20).

**File custom yang dipakai**:
| peran | file | ukuran |
|---|---|---|
| diffusion (unet) | `qwen_image_2.1_int8_convrot.safetensors` (INT8 convrot, native T4) | 7.26 GB |
| text encoder | `Qwen3VL-8B-Instruct-Q4_K_M.gguf` | 5.03 GB |
| vision (edit saja) | `mmproj-Qwen3VL-8B-Instruct-Q8_0.gguf` → `--llm_vision` | 0.75 GB |
| vae | `vae/qwen_image_2.1_vae_bf16.safetensors` | 0.68 GB |
| LoRA turbo (opsional) | `Qwen-Image-2.1-viggle-turbo-v0.3-6step-lora-r128.safetensors` | 0.68 GB |

**Mode Turbo** (checkbox di cell ⑥, default nyala): pakai LoRA r128 Viggle v0.3
unmerged (sd.cpp apply at runtime — persis yang direkomendasikan Viggle,
jangan di-merge). Resep resmi: **6 step**,
`--sigmas "1.0,0.9375,0.875,0.75,0.5,0.25"`, **cfg 1.0** (no CFG),
tanpa negative prompt, LoRA strength 1.0. Base unet adalah
**upstream base biasa (bukan turbo-merged)** → LoRA ini yang bikin kenceng.

**Yang perlu kamu tahu:**
- Release resmi **tidak ada binary Linux CUDA** (cuma Windows CUDA) → cell ③
  download binary prebuilt (hasil build GitHub Actions sendiri, CUDA 12.6,
  ~718 MB, sha256 diverifikasi). Sekali per runtime (±2–5 mnt).
- T4 = compute capability 7.5 → jalur native INT8 convrot CUDA kepakai
  (syaratnya persis 7.5+). Tanpa Python/torch sama sekali — murni `sd-cli`.
- Total weight **±14.4 GB < VRAM 15 GB** → muat penuh di VRAM; checkbox
  `offload_to_cpu` tetap ada tapi **default mati** — TERBUKTI di T4:
  nyala = proses dibunuh OOM killer (RAM Colab cuma 12.7 GB < bobot
  13.4 GB). Nyalain cuma kalau VRAM lo lebih kecil dari T4.
- TE GGUF = text-only → mode **Edit** wajib pasang `--llm_vision` (mmproj,
  otomatis di cell ⑥); mode T2I tidak perlu.
- Dimensi gambar **harus kelipatan 32** (ditangani otomatis di cell ⑥).

**Urutan cell:** ① cek GPU → ② install downloader → ③ download sd-cli prebuilt →
④ download model → ⑤ upload referensi (opsional) → ⑥ generate → ⑦ batch (opsional).

> Cell ③ & ④ punya guard: re-run tidak akan re-download.


> **Catatan:** resep ini (flag, turbo 6-step, VRAM fit) verified jalan di
> Colab T4 gratis (txt2img ±96 detik). Varian model resmi di atas memakai
> arsitektur & kuantisasi yang sama dengan yang dites, tapi belum diuji
> ulang di GPU.


In [ ]:
#@title ① Cek GPU / disk { vertical-output: true }
import shutil, subprocess
print("=== nvidia-smi ===")
r = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                    "--format=csv,noheader"], capture_output=True, text=True)
print(r.stdout.strip() or "nvidia-smi tidak ada — pilih Runtime > Change runtime type > GPU!")
free_gb = shutil.disk_usage("/content").free / 1e9
print(f"disk /content bebas: {free_gb:.1f} GB")
need = 20.0
print(("OK, cukup untuk model (14.4 GB) + sd-cli." if free_gb >= need
       else f"WARNING: butuh ~{need:.0f} GB bebas, sekarang cuma {free_gb:.1f} GB!"))



In [ ]:
#@title ② Install downloader (sekali per runtime) { vertical-output: true }
# Tidak butuh torch/transformers maupun compiler sama sekali — sd-cli sudah
# jadi binary prebuilt, yang dibutuhkan cuma downloader (aria2 buat model).
!apt-get update -qq && apt-get install -y -qq aria2 > /dev/null 2>&1
!aria2c --version | head -1



In [ ]:
#@title ③ Download sd-cli prebuilt (CUDA 12.6) { vertical-output: true }
#@markdown Sekali per runtime (±2–5 mnt). Binary hasil build GitHub Actions
#@markdown (sd.cpp sdcpp-master-929-3f8527a-cuda12.6, CUDA 12.6, sm_75;80;86;89;90,
#@markdown CUDA runtime dibundle di lib/). Kalau sudah ada, cell ini langsung skip.
%%bash
set -e
URL="https://github.com/fendimaharsi0-sketch/sdcpp-cuda-linux/releases/download/sdcpp-master-929-3f8527a-cuda12.6/sdcpp-sd-cli-cuda12.6-ubuntu22.04-x86_64.tar.gz"
SHA="1b19a8815c2f06ed36f963648d621ea593a54f2778df09dbb21e3ead781b45eb"
DEST="/content/sdcpp-cuda"
SH="$DEST/sd-cli.sh"
if [ -x "$SH" ]; then
  echo "sd-cli sudah ada — skip download."
  "$SH" --help 2>&1 | head -3
  exit 0
fi
mkdir -p "$DEST"
cd "$DEST"
echo "== download prebuilt sd-cli (~718 MB) =="
wget -q --show-progress -O sdcpp.tar.gz "$URL"
echo "== verifikasi sha256 =="
echo "$SHA  sdcpp.tar.gz" | sha256sum -c -
echo "== extract =="
tar xzf sdcpp.tar.gz --no-same-owner
rm sdcpp.tar.gz
chmod +x "$DEST/sd-cli.sh" "$DEST/bin/sd-cli"
echo "== selesai =="
ls -la "$DEST/bin/sd-cli" "$DEST/lib"
"$SH" --help 2>&1 | head -3



In [ ]:
#@title ④ Download model (14.4 GB — auto skip kalau sudah ada) { vertical-output: true }
import os, subprocess
# (repo, nama file di repo, ukuran byte eksak dari HF API, folder lokal)
FILES = [
    ("Comfy-Org/Qwen-Image-2.1",
     "diffusion_models/qwen_image_2.1_int8_convrot.safetensors", 7256783064, "/content/models/diffusion"),
    ("Qwen/Qwen3-VL-8B-Instruct-GGUF",
     "Qwen3VL-8B-Instruct-Q4_K_M.gguf", 5027784800, "/content/models/te"),
    ("Qwen/Qwen3-VL-8B-Instruct-GGUF",
     "mmproj-Qwen3VL-8B-Instruct-Q8_0.gguf", 752289728, "/content/models/te"),
    ("Comfy-Org/Qwen-Image-2.1",
     "vae/qwen_image_2.1_vae_bf16.safetensors", 675509688, "/content/models/vae"),
    ("Viggle/Qwen-Image-2.1-viggle-turbo",
     "Qwen-Image-2.1-viggle-turbo-v0.3-6step-lora-r128.safetensors", 679604800, "/content/models/lora"),
]
for repo, fname, size, d in FILES:
    os.makedirs(d, exist_ok=True)
    base = os.path.basename(fname)
    p = os.path.join(d, base)
    if os.path.exists(p) and os.path.getsize(p) == size:
        print(f"OK skip {base} ({size/1e9:.2f} GB sudah ada)")
        continue
    url = f"https://huggingface.co/{repo}/resolve/main/{fname}"
    print(f"download {base} ({size/1e9:.2f} GB) ...")
    subprocess.run(["aria2c", "-x", "16", "-s", "16", "-c", "--console-log-level=warn",
                    "--summary-interval=30", "-d", d, "-o", base, url], check=True)
    got = os.path.getsize(p)
    assert got == size, f"SIZE MISMATCH {base}: {got} != {size}, hapus & ulangi cell ini"
    print(f"OK {base}")
print("\nSemua model siap.")



In [ ]:
#@title ⑤ Upload gambar referensi (opsional — untuk mode Edit) { vertical-output: true }
#@markdown Upload 1–2 gambar, lalu tulis **nama file-nya** di param `ref1`/`ref2`
#@markdown cell ⑥. Format: jpg/png/webp/bmp.
import os
os.makedirs("/content/refs", exist_ok=True)
from google.colab import files
up = files.upload()
for name, data in up.items():
    open(f"/content/refs/{name}", "wb").write(data)
    print("saved:", name)
print()
!ls -la /content/refs



In [ ]:
#@title ⑥ Generate { vertical-output: true }
#@markdown ### Mode & prompt
mode = "Text-to-Image" #@param ["Text-to-Image", "Edit (1-2 ref)"]
prompt = "a lovely cat holding a sign that says 'qwen2.1.cpp'" #@param {type:"string"}
negative_prompt = "" #@param {type:"string"}
#@markdown ### Referensi (mode Edit saja — nama file di /content/refs)
ref1 = "" #@param {type:"string"}
ref2 = "" #@param {type:"string"}
#@markdown ### Turbo (LoRA Viggle v0.3, 6 step)
turbo = True #@param {type:"boolean"}
#@markdown ### Ukuran & sampling (diabaikan kalau turbo nyala)
auto_size_ref1 = True #@param {type:"boolean"}
width = 1024 #@param {type:"integer"}
height = 1024 #@param {type:"integer"}
steps = 20 #@param {type:"integer"}
cfg_scale = 6.0 #@param {type:"number"}
sampler = "euler" #@param ["euler", "euler_a", "heun", "dpm2", "dpmpp_2m", "dpmpp_2s_a", "lcm"]
seed = -1 #@param {type:"integer"}
#@markdown ### Memori
offload_to_cpu = False #@param {type:"boolean"}
prefix_cache = "auto" #@param ["auto", "q8_0", "off"]
te_offload_cpu = False #@param {type:"boolean"}
#@markdown ### Output
out_name = "qwen21_output.png" #@param {type:"string"}

import os, subprocess, random
from PIL import Image as PILImage
from IPython.display import Image, display

SD = "/content/sdcpp-cuda/sd-cli.sh"  # wrapper: set LD_LIBRARY_PATH ke lib/ bawaan
DIFF = "/content/models/diffusion/qwen_image_2.1_int8_convrot.safetensors"
LLM  = "/content/models/te/Qwen3VL-8B-Instruct-Q4_K_M.gguf"
MMPROJ = "/content/models/te/mmproj-Qwen3VL-8B-Instruct-Q8_0.gguf"
VAE  = "/content/models/vae/qwen_image_2.1_vae_bf16.safetensors"
LORA = "/content/models/lora/Qwen-Image-2.1-viggle-turbo-v0.3-6step-lora-r128.safetensors"
for p, label in [(SD, "sd-cli"), (DIFF, "diffusion"), (LLM, "text encoder"), (VAE, "vae")]:
    assert os.path.exists(p), f"{label} belum ada di {p} — jalankan cell ③/④ dulu!"

# --- mode turbo: resep resmi Viggle v0.3 (LoRA unmerged, 6 step, cfg 1.0) ---
TURBO_SIGMAS = "1.0,0.9375,0.875,0.75,0.5,0.25"
if turbo:
    assert os.path.exists(LORA), f"LoRA turbo belum ada di {LORA} — jalankan cell ④ dulu!"
    prompt = prompt.strip() + f" <lora:{os.path.basename(LORA)}:1.0>"
    steps, cfg_scale, sampler = 6, 1.0, "euler"
    if negative_prompt.strip():
        print("note: turbo tidak pakai negative prompt — diabaikan.")
        negative_prompt = ""
    print("TURBO ON: 6 step, sigmas Viggle, cfg 1.0, LoRA r128 @1.0 (runtime)")

def snap32(x):
    return max(32, (int(x) // 32) * 32)

def guard_ref(p):
    # batasi input ref ke ~1MP (LANCZOS, keep aspect) biar KV/aktivasi gak meledak
    im = PILImage.open(p)
    w, h = im.size
    if w * h > 1024 * 1024:
        s = (1024 * 1024 / (w * h)) ** 0.5
        nw, nh = int(w * s), int(h * s)
        im = im.resize((nw, nh), PILImage.LANCZOS)
        os.makedirs("/content/refs_guarded", exist_ok=True)
        g = os.path.join("/content/refs_guarded", os.path.basename(p))
        im.save(g)
        print(f"guard input: {os.path.basename(p)} {w}x{h} -> {nw}x{nh}")
        return g
    return p

refs = []
if mode.startswith("Edit"):
    for r in (ref1.strip(), ref2.strip()):
        if r:
            p = os.path.join("/content/refs", r)
            assert os.path.exists(p), f"ref tidak ketemu: {p} (upload dulu di cell ⑤)"
            refs.append(guard_ref(p))
    assert refs, "mode Edit butuh minimal 1 ref — isi ref1!"
    if auto_size_ref1:
        # canvas ngikutin aspect ratio ref1, ~1MP, kelipatan 32
        w0, h0 = PILImage.open(refs[0]).size
        s = (1024 * 1024 / (w0 * h0)) ** 0.5
        width, height = snap32(w0 * s), snap32(h0 * s)
        print(f"auto size dari ref1: {w0}x{h0} -> {width}x{height}")
    # TE GGUF = text-only -> mode Edit butuh vision tower via --llm_vision
    assert os.path.exists(MMPROJ), f"mmproj belum ada di {MMPROJ} — jalankan cell ④ dulu!"
else:
    width, height = snap32(width), snap32(height)

if seed < 0:
    seed = random.randint(0, 2**31 - 1)
print(f"seed: {seed}")

os.makedirs("/content/outputs", exist_ok=True)
out_path = os.path.join("/content/outputs", out_name)

cmd = [SD,
       "--diffusion-model", DIFF,
       "--llm", LLM,
       "--vae", VAE,
       "--lora-model-dir", "/content/models/lora",  # dibaca via <lora:nama:1.0> di prompt
       "-p", prompt,
       "-W", str(width), "-H", str(height),
       "--steps", str(steps),
       "--cfg-scale", str(cfg_scale),
       "--sampling-method", sampler,
       "-s", str(seed),
       "--fa",                # flash attention (CUDA)
       "-v"]
if turbo:
    cmd += ["--sigmas", TURBO_SIGMAS]
if te_offload_cpu:
    # TE (5.0 GB) pindah ke CPU: hemat ~5.0 GB VRAM (RAM 12.7 GB aman),
    # berguna buat mode Edit yang lebih boros. Perlu verifikasi flag di T4.
    cmd += ["--backend", "diffusion=cuda0,te=cpu",
            "--params-backend", "diffusion=cuda0,te=cpu"]
if offload_to_cpu:
    # HATI-HATI: di Colab T4 ini TERBUKTI bikin proses dibunuh OOM killer
    # (RAM 12.7 GB < total bobot 14.4 GB). Cuma nyalain kalau VRAM < 14 GB.
    cmd += ["--offload-to-cpu"]
if negative_prompt.strip():
    cmd += ["-n", negative_prompt]
for r in refs:
    cmd += ["-r", r]
if mode.startswith("Edit"):
    cmd += ["--llm_vision", MMPROJ]  # vision tower, wajib untuk Edit
if prefix_cache == "q8_0":
    cmd += ["--model-args", "qwen_image_2_1_prefix_cache_type=q8_0"]
elif prefix_cache == "off":
    cmd += ["--model-args", "qwen_image_2_1_prefix_cache=false"]
cmd += ["-o", out_path]

print("\n$ " + " ".join(f"'{c}'" if " " in c else c for c in cmd) + "\n")
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
rc = proc.wait()
assert rc == 0 and os.path.exists(out_path), f"generate gagal (rc={rc})"
print("\n=== HASIL:", out_path, "===")
display(Image(out_path))



### ⑦ Batch (opsional)

Ulangi generate dengan seed berbeda tanpa ubah param lain. Edit `n` lalu run.



In [ ]:
#@title ⑦ Batch — N variasi seed berbeda { vertical-output: true }
#@markdown Menjalankan ulang perintah cell ⑥ dengan seed acak berbeda.
#@markdown Pastikan cell ⑥ sudah pernah sukses sekali sebelum pakai ini.
n = 4 #@param {type:"integer"}

import subprocess, random, os
from IPython.display import Image, display

# ambil param terakhir dari cell ⑥ (variabel masih ada di namespace)
batch_seeds = [random.randint(0, 2**31 - 1) for _ in range(n)]
base_cmd = cmd[:-2]  # cmd dari cell ⑥, tanpa "-o out_path"
base_out = out_path.rsplit(".", 1)
for i, sd in enumerate(batch_seeds):
    op = f"{base_out[0]}_batch{i+1}.{base_out[1]}"
    c = [a for a in base_cmd]
    si = c.index("-s")
    c[si + 1] = str(sd)
    c += ["-o", op]
    print(f"\n--- batch {i+1}/{n} seed={sd} ---")
    r = subprocess.run(c, capture_output=True, text=True)
    print(r.stdout[-1500:] if r.stdout else "")
    if r.returncode == 0 and os.path.exists(op):
        display(Image(op))
    else:
        print("GAGAL:", r.stderr[-500:] if r.stderr else "?")
        break

